In [1]:
import torch
import gc

# Очищаем кэш GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


# Принудительный сборщик мусора
gc.collect()

# Проверяем результат
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU memory allocated: 0.00 GB
GPU memory cached: 0.00 GB


In [2]:
import sys
import os
import torch
import numpy as np
from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoTokenizer

import torch_pruning as tp

RESULTS_DIR = "DG_research/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

BATCH_SIZE = 2
SEQ_LENGTH = 64
# MODEL_NAME = "google/gemma-2-2b"
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Model loaded")

Model loaded


In [9]:
example_inputs = {
    "input_ids": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device),
    "attention_mask": torch.ones(BATCH_SIZE, SEQ_LENGTH).to(device),
    "labels": torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, SEQ_LENGTH)).to(device)
}

In [10]:
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

wrapped_model = ModelWrapper(model).to(device)

DG_before = tp.DependencyGraph().build_dependency(
    wrapped_model,
    example_inputs=example_inputs
)

print("DependencyGraph built")

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.model.layers.2.post_attention_layernorm.weight', 'model.model.layers.3.post_attention_layernorm.weight', 'model.model.layers.4.input_layernorm.weight', 'model.model.layers.18.post_attention_layernorm.weight', 'model.model.layers.23.input_layernorm.weight', 'model.model.layers.10.post_attention_layernorm.weight', 'model.model.layers.11.input_layernorm.weight', 'model.model.layers.15.post_attention_layernorm.weight', 'model.model.layers.16.input_layernorm.weight', 'model.model.layers.5.post_attention_layernorm.weight', 'model.model.layers.6.post_attention_layernorm.weight', 'model.model.layers.7.input_layernorm.weight', 'model.model.layers.12.post_attention_layernorm.weight', 'model.model.layers.13.input_layernorm.weight', 'model.model.layers.21.input_layernorm.weight', 'model.model.layers.22.input_layernorm.weight', 'model.model.layers

DependencyGraph built


In [11]:
pruning_ratio = 0.3  # примеры, можно варьировать
ignored_layers = []

# игнорируем embedding и lm_head
for name, module in model.named_modules():
    if 'embed_tokens' in name or 'lm_head' in name:
        ignored_layers.append(module)

pruner = tp.pruner.MetaPruner(
    model=wrapped_model,
    example_inputs=example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2),
    pruning_ratio=pruning_ratio,
    ignored_layers=ignored_layers,
    round_to=8,
    iterative_steps=1
)

/home/user/projects/FedCore/.venv/lib/python3.10/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.model.layers.2.post_attention_layernorm.weight', 'model.model.layers.3.post_attention_layernorm.weight', 'model.model.layers.4.input_layernorm.weight', 'model.model.layers.18.post_attention_layernorm.weight', 'model.model.layers.23.input_layernorm.weight', 'model.model.layers.10.post_attention_layernorm.weight', 'model.model.layers.11.input_layernorm.weight', 'model.model.layers.15.post_attention_layernorm.weight', 'model.model.layers.16.input_layernorm.weight', 'model.model.layers.5.post_attention_layernorm.weight', 'model.model.layers.6.post_attention_layernorm.weight', 'model.model.layers.7.input_layernorm.weight', 'model.model.layers.12.post_attention_layernorm.weight', 'model.model.layers.13.input_layernorm.weight', 'model.model.layers.21.input_layernorm.weight', 'model.model.layers.22.input_layernorm.weight', 'model.model.layers

In [ ]:
pruner.step()
pruned_model = wrapped_model.model

In [ ]:
DG_after = tp.DependencyGraph().build_dependency(ModelWrapper(pruned_model).to(device), example_inputs=example_inputs)
print("DependencyGraph rebuilt (after pruning)")

In [ ]:
def extract_group_info(pruner, model, tag):

    module_to_name = {m: name for name, m in model.named_modules()}
    module_to_params = {m: list(m.parameters(recurse=False)) for m in model.modules()}

    groups = pruner.DG.get_all_groups()

    rows = []

    for i, g in enumerate(groups):

        total_params = 0
        lora_params = 0
        base_params = 0
        module_types = defaultdict(int)

        for dep, idxs in g:
            target = dep.target.module

            if isinstance(target, torch.nn.Module):
                module_types[target.__class__.__name__] += 1

                for p in module_to_params.get(target, []):
                    total_params += p.numel()

                    name = module_to_name.get(target, "")
                    if "lora_" in name:
                        lora_params += p.numel()
                    else:
                        base_params += p.numel()

        # importance
        try:
            score = pruner.importance(g).item()
        except:
            score = 0.0

        rows.append({
            "group_id": i,
            "total_params": total_params,
            "lora_params": lora_params,
            "base_params": base_params,
            "lora_ratio": lora_params / max(1, total_params),
            "importance": score,
            "stage": tag
        })

    return pd.DataFrame(rows)

In [ ]:
df_groups, df_intersections = analyze_dependency_graph_fast(
    DG,
    model,
    results_dir=RESULTS_DIR
)

In [40]:
df = pd.read_csv("DG_research/results/dg_intersections.csv")
df.head()

,group_i,group_j,intersection_size,union_size,overlap_ratio
0,0,1,2,7,0.285714
1,0,2,2,7,0.285714
2,0,3,7,290,0.024138
3,0,6,7,290,0.024138
4,0,10,7,290,0.024138
